# 🧠 MCQ Bubble Sheet Recognition using LeCun's CNN (LeNet)

---

## **📌 Project Overview**
This notebook trains a **Convolutional Neural Network (CNN)** based on **Yann LeCun's LeNet architecture** to automatically recognize and classify answers from **MCQ bubble sheets**. The model is trained on a large-scale synthetic dataset of **420,000+ images** generated programmatically in Python.

---

## **📂 Dataset Description**
The dataset is organized into **3 main folders**:

- **🗂️ single_questions** — Contains **400,000 images** of individual bubble cells
  - 100 questions (Q1–Q100) × 4 answer options (A/B/C/D) × 1000 images each
  - Each image shows a single filled or unfilled bubble option

- **🗂️ pair_of_10** — Contains **10,000 images** of 10-question strips
  - Each image shows a row of 10 MCQ questions together

- **🗂️ pair_of_20** — Contains **10,000 images** of 20-question strips
  - Each image shows a row of 20 MCQ questions together

Each image folder has a corresponding **labels folder** with JSON annotation files.

---

## **🏗️ Model Architecture — LeCun's LeNet**
We follow the classic **LeNet-5 (1998)** design by Yann LeCun with the following key principles:

- ✅ **Grayscale input** normalized to **[-1, 1]**
- ✅ **Average Pooling** (subsampling layers — original LeCun style)
- ✅ **Tanh activations** throughout the network
- ✅ **C1 → S2 → C3 → S4 → C5 → F6 → Output** layer structure
- ✅ Three separate models trained for **single, 10-pair, and 20-pair** images

---

## **⚡ Training Strategy**
- **Train/Test Split:** 80% training — 20% testing
- **Optimizer:** Adam (lr = 1e-3) with StepLR scheduler
- **Loss Function:** Cross Entropy Loss
- **Batch Size:** 256
- **Epochs:** 10
- **Mixed Precision:** ✅ Enabled via `torch.cuda.amp` for faster GPU training
- **Hardware:** Kaggle GPU (T4 / P100)

---

## **📊 Results Summary**

| **Model** | **Train Acc** | **Val Acc** |
|---|---|---|
| 🟢 Single Question LeNet | 100.00% | 100.00% |
| 🟢 Pair-10 LeNet | 100.00% | 100.00% |
| 🟢 Pair-20 LeNet | 99.96% | 99.86% |

---

## **💾 Saved Models**
All trained models are saved to `/kaggle/working/`:
- 📦 `lenet_single.pth`
- 📦 `lenet_pair10.pth`
- 📦 `lenet_pair20.pth`

---

## **🔭 Future Work**
- 🔹 Test on **real scanned bubble sheets** to evaluate real-world generalization
- 🔹 Add **data augmentation** (noise, blur, rotation) for robustness
- 🔹 Build a **full inference pipeline** for end-to-end bubble sheet grading
- 🔹 Extend to detect **partially filled** or **multiple marked** bubbles

---

> **📝 Note:** The dataset used in this notebook is fully synthetic and was generated using Python. 
> Results may vary on real-world scanned answer sheets due to noise, lighting, and scanning artifacts.

# **📌 Install and Imports**

In [1]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
from torch.cuda.amp import GradScaler, autocast
from PIL import Image
from pathlib import Path
import glob
from concurrent.futures import ThreadPoolExecutor
import multiprocessing
import warnings

warnings.filterwarnings('ignore')

print(f"GPU Available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

GPU Available: True
Device: Tesla T4


# **📌 Fast Path Indexing**

In [2]:
BASE = "/kaggle/input/mcqs-bubble-sheet-420k-images"

SINGLE_IMG  = f"{BASE}/single_questions/single_questions/images"
SINGLE_LBL  = f"{BASE}/single_questions/single_questions/labels"
P10_IMG     = f"{BASE}/pair_of_10/pair_of_10/images"
P10_LBL     = f"{BASE}/pair_of_10/pair_of_10/labels"
P20_IMG     = f"{BASE}/pair_of_20/pair_of_20/images"
P20_LBL     = f"{BASE}/pair_of_20/pair_of_20/labels"

def build_single_index(img_root, lbl_root):
    """
    Walk single_questions structure:
    images/Q{i}/{A,B,C,D}/xxxx.png  ->  labels/Q{i}/{A,B,C,D}/xxxx.json
    Returns list of (img_path, lbl_path, answer_idx)
    answer map: A=0, B=1, C=2, D=3
    """
    answer_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
    index = []
    img_root = Path(img_root)
    lbl_root = Path(lbl_root)

    # Use glob once — much faster than nested os.walk
    img_paths = list(img_root.glob("Q*/*/*.png")) + list(img_root.glob("Q*/*/*.jpg"))
    print(f"  Found {len(img_paths)} single question images")

    for img_p in img_paths:
        # img_p: .../images/Q5/B/0042.png
        rel = img_p.relative_to(img_root)          # Q5/B/0042.png
        lbl_p = lbl_root / rel.parent / (rel.stem + ".json")
        answer_key = rel.parts[1]                   # 'A','B','C','D'
        if answer_key in answer_map and lbl_p.exists():
            index.append((str(img_p), str(lbl_p), answer_map[answer_key]))

    return index

def build_pair_index(img_root, lbl_root):
    """
    Walk pair_of_10 / pair_of_20:
    images/xxxx.png  ->  labels/xxxx.json
    Label has list of answers for each question in the pair.
    Returns list of (img_path, lbl_path)
    """
    img_root = Path(img_root)
    lbl_root = Path(lbl_root)
    img_paths = list(img_root.glob("*.png")) + list(img_root.glob("*.jpg"))
    print(f"  Found {len(img_paths)} pair images")

    index = []
    for img_p in img_paths:
        lbl_p = lbl_root / (img_p.stem + ".json")
        if lbl_p.exists():
            index.append((str(img_p), str(lbl_p)))
    return index

print("Building indexes (fast, no image loading)...")
single_index = build_single_index(SINGLE_IMG, SINGLE_LBL)
p10_index    = build_pair_index(P10_IMG, P10_LBL)
p20_index    = build_pair_index(P20_IMG, P20_LBL)

print(f"\nIndex sizes -> Single: {len(single_index)} | P10: {len(p10_index)} | P20: {len(p20_index)}")

Building indexes (fast, no image loading)...
  Found 400000 single question images
  Found 10000 pair images
  Found 10000 pair images

Index sizes -> Single: 400000 | P10: 10000 | P20: 10000


# **📌 Dataset Classes**

In [3]:
ANSWER_MAP = {'A': 0, 'B': 1, 'C': 2, 'D': 3}

# ------- transforms (LeCun-style: grayscale, normalize to [-1,1]) -------
def get_transform(img_size=(32, 32)):
    return transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize(img_size),
        transforms.ToTensor(),                        # [0,1]
        transforms.Normalize(mean=[0.5], std=[0.5])  # [-1,1]  LeCun style
    ])

transform_single = get_transform((32, 32))   # single bubble cell
transform_pair10 = get_transform((64, 128))  # 10-question strip  (H x W)
transform_pair20 = get_transform((64, 256))  # 20-question strip

# ---- helper: read answer from json ----
def read_single_label(path):
    with open(path) as f:
        data = json.load(f)
    # expected: {"answer": "B"} or {"label": "B"} or {"answer": 1}
    val = data.get("answer", data.get("label", None))
    if isinstance(val, str):
        return ANSWER_MAP.get(val.upper(), 0)
    return int(val)

def read_pair_label(path, n_questions):
    with open(path) as f:
        data = json.load(f)
    # expected: {"answers": ["A","C","B",...]} list of length n_questions
    answers = data.get("answers", data.get("labels", []))
    return torch.tensor([ANSWER_MAP.get(a.upper(), 0) if isinstance(a, str)
                         else int(a) for a in answers[:n_questions]], dtype=torch.long)

# ---- Datasets ----
class SingleQuestionDataset(Dataset):
    def __init__(self, index, transform):
        self.index = index
        self.transform = transform

    def __len__(self): return len(self.index)

    def __getitem__(self, idx):
        img_p, lbl_p, _ = self.index[idx]   # answer_idx from folder name
        img = Image.open(img_p).convert("RGB")
        label = read_single_label(lbl_p)     # verify from json too
        return self.transform(img), torch.tensor(label, dtype=torch.long)


class PairDataset(Dataset):
    def __init__(self, index, transform, n_questions):
        self.index = index
        self.transform = transform
        self.n_q = n_questions

    def __len__(self): return len(self.index)

    def __getitem__(self, idx):
        img_p, lbl_p = self.index[idx]
        img = Image.open(img_p).convert("RGB")
        labels = read_pair_label(lbl_p, self.n_q)
        return self.transform(img), labels

# **📌 Create Datasets & Dataloaders**

In [4]:
# Cell 4 (FIXED): Create Datasets & DataLoaders

NUM_WORKERS = 2      # reduced — Kaggle shared memory is limited
PIN_MEMORY  = True
BATCH_SIZE  = 256

ds_single = SingleQuestionDataset(single_index, transform_single)
ds_p10    = PairDataset(p10_index,  transform_pair10, n_questions=10)
ds_p20    = PairDataset(p20_index,  transform_pair20, n_questions=20)

def split_dataset(ds, ratio=0.8):
    n_train = int(len(ds) * ratio)
    n_test  = len(ds) - n_train
    return random_split(ds, [n_train, n_test],
                        generator=torch.Generator().manual_seed(42))

train_single, test_single = split_dataset(ds_single)
train_p10,    test_p10    = split_dataset(ds_p10)
train_p20,    test_p20    = split_dataset(ds_p20)

def make_loader(ds, shuffle=True):
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=False,   # <-- FIXED: was True, causes crash on Kaggle
        prefetch_factor=2 if NUM_WORKERS > 0 else None
    )

loader_train_single = make_loader(train_single)
loader_test_single  = make_loader(test_single, shuffle=False)
loader_train_p10    = make_loader(train_p10)
loader_test_p10     = make_loader(test_p10, shuffle=False)
loader_train_p20    = make_loader(train_p20)
loader_test_p20     = make_loader(test_p20, shuffle=False)

print(f"Single -> Train: {len(train_single):,}  Test: {len(test_single):,}")
print(f"P10    -> Train: {len(train_p10):,}  Test: {len(test_p10):,}")
print(f"P20    -> Train: {len(train_p20):,}  Test: {len(test_p20):,}")

Single -> Train: 320,000  Test: 80,000
P10    -> Train: 8,000  Test: 2,000
P20    -> Train: 8,000  Test: 2,000


# **📌 LeCun-Style CNN Models**

In [5]:
# ---- Model A: Single Question (LeNet-5 variant) ----
class LeNetSingle(nn.Module):
    """
    LeCun 1998 LeNet-5 adapted for 1x32x32 input, 4-class output (A/B/C/D)
    """
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            # C1: 1x32x32 -> 6x28x28
            nn.Conv2d(1, 6, kernel_size=5),
            nn.Tanh(),
            # S2: 6x28x28 -> 6x14x14  (avg pool = LeCun subsampling)
            nn.AvgPool2d(2, 2),

            # C3: 6x14x14 -> 16x10x10
            nn.Conv2d(6, 16, kernel_size=5),
            nn.Tanh(),
            # S4: 16x10x10 -> 16x5x5
            nn.AvgPool2d(2, 2),

            # C5: 16x5x5 -> 120x1x1
            nn.Conv2d(16, 120, kernel_size=5),
            nn.Tanh(),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(120, 84),
            nn.Tanh(),
            nn.Linear(84, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# ---- Model B: Pair Model (deeper LeNet for wider input) ----
class LeNetPair(nn.Module):
    """
    Adapted for 1xHxW strip images containing N questions.
    Outputs N * 4 logits -> reshape to [B, N, 4] for per-question loss.
    """
    def __init__(self, n_questions=10, in_h=64, in_w=128):
        super().__init__()
        self.n_q = n_questions

        self.features = nn.Sequential(
            nn.Conv2d(1, 6,  kernel_size=5, padding=2), nn.Tanh(),
            nn.AvgPool2d(2, 2),                          # H/2, W/2

            nn.Conv2d(6, 16, kernel_size=5, padding=2), nn.Tanh(),
            nn.AvgPool2d(2, 2),                          # H/4, W/4

            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.Tanh(),
            nn.AvgPool2d(2, 2),                          # H/8, W/8
        )

        # compute flattened size
        dummy = torch.zeros(1, 1, in_h, in_w)
        flat  = self.features(dummy).numel()

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat, 256), nn.Tanh(),
            nn.Linear(256, n_questions * 4),  # 4 classes per question
        )

    def forward(self, x):
        out = self.classifier(self.features(x))
        return out.view(out.size(0), self.n_q, 4)  # [B, N, 4]


model_single = LeNetSingle(num_classes=4).to(DEVICE)
model_p10    = LeNetPair(n_questions=10, in_h=64,  in_w=128).to(DEVICE)
model_p20    = LeNetPair(n_questions=20, in_h=64,  in_w=256).to(DEVICE)

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"LeNetSingle params : {count_params(model_single):,}")
print(f"LeNetPair-10 params: {count_params(model_p10):,}")
print(f"LeNetPair-20 params: {count_params(model_p20):,}")

LeNetSingle params : 61,196
LeNetPair-10 params: 1,066,324
LeNetPair-20 params: 2,125,180


# **📌 Training Utilities**

In [6]:
def train_single_epoch(model, loader, optimizer, scaler, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            out  = model(imgs)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


def eval_single(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            with autocast():
                out  = model(imgs)
                loss = criterion(out, labels)
            total_loss += loss.item() * imgs.size(0)
            correct    += (out.argmax(1) == labels).sum().item()
            total      += imgs.size(0)
    return total_loss / total, correct / total


def train_pair_epoch(model, loader, optimizer, scaler, criterion, n_q):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        # labels: [B, N]
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            out  = model(imgs)              # [B, N, 4]
            # reshape for CrossEntropy: [B*N, 4] vs [B*N]
            loss = criterion(out.view(-1, 4), labels.view(-1))
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(-1) == labels).sum().item()
        total      += imgs.size(0) * n_q
    return total_loss / (total / n_q), correct / total


def eval_pair(model, loader, criterion, n_q):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            with autocast():
                out  = model(imgs)
                loss = criterion(out.view(-1, 4), labels.view(-1))
            total_loss += loss.item() * imgs.size(0)
            correct    += (out.argmax(-1) == labels).sum().item()
            total      += imgs.size(0) * n_q
    return total_loss / (total / n_q), correct / total

# **📌 Train Single Questions Model**

In [7]:
EPOCHS = 10
criterion = nn.CrossEntropyLoss()

opt_s    = Adam(model_single.parameters(), lr=1e-3)
sched_s  = StepLR(opt_s, step_size=7, gamma=0.1)
scaler_s = GradScaler()

print("=" * 55)
print("Training: Single Question LeNet")
print("=" * 55)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_single_epoch(model_single, loader_train_single,
                                         opt_s, scaler_s, criterion)
    vl_loss, vl_acc = eval_single(model_single, loader_test_single, criterion)
    sched_s.step()
    print(f"Ep {epoch:02d}/{EPOCHS} | "
          f"Train Loss: {tr_loss:.4f}  Acc: {tr_acc*100:.2f}% | "
          f"Val Loss:   {vl_loss:.4f}  Acc: {vl_acc*100:.2f}%")

torch.save(model_single.state_dict(), "/kaggle/working/lenet_single.pth")
print("Model saved.")

Training: Single Question LeNet
Ep 01/10 | Train Loss: 0.0145  Acc: 99.78% | Val Loss:   0.0000  Acc: 100.00%
Ep 02/10 | Train Loss: 0.0000  Acc: 100.00% | Val Loss:   0.0000  Acc: 100.00%
Ep 03/10 | Train Loss: 0.0000  Acc: 100.00% | Val Loss:   0.0000  Acc: 100.00%
Ep 04/10 | Train Loss: 0.0000  Acc: 100.00% | Val Loss:   0.0000  Acc: 100.00%
Ep 05/10 | Train Loss: 0.0000  Acc: 100.00% | Val Loss:   0.0000  Acc: 100.00%
Ep 06/10 | Train Loss: 0.0000  Acc: 100.00% | Val Loss:   0.0000  Acc: 100.00%
Ep 07/10 | Train Loss: 0.0000  Acc: 100.00% | Val Loss:   0.0000  Acc: 100.00%
Ep 08/10 | Train Loss: 0.0000  Acc: 100.00% | Val Loss:   0.0000  Acc: 100.00%
Ep 09/10 | Train Loss: 0.0000  Acc: 100.00% | Val Loss:   0.0000  Acc: 100.00%
Ep 10/10 | Train Loss: 0.0000  Acc: 100.00% | Val Loss:   0.0000  Acc: 100.00%
Model saved.


# **📌 Train Pair-10 Model**

In [8]:
opt_p10    = Adam(model_p10.parameters(), lr=1e-3)
sched_p10  = StepLR(opt_p10, step_size=7, gamma=0.1)
scaler_p10 = GradScaler()

print("=" * 55)
print("Training: Pair-10 LeNet")
print("=" * 55)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_pair_epoch(model_p10, loader_train_p10,
                                       opt_p10, scaler_p10, criterion, n_q=10)
    vl_loss, vl_acc = eval_pair(model_p10, loader_test_p10, criterion, n_q=10)
    sched_p10.step()
    print(f"Ep {epoch:02d}/{EPOCHS} | "
          f"Train Loss: {tr_loss:.4f}  Acc: {tr_acc*100:.2f}% | "
          f"Val Loss:   {vl_loss:.4f}  Acc: {vl_acc*100:.2f}%")

torch.save(model_p10.state_dict(), "/kaggle/working/lenet_pair10.pth")
print("Model saved.")

Training: Pair-10 LeNet
Ep 01/10 | Train Loss: 1.0356  Acc: 67.95% | Val Loss:   0.3476  Acc: 97.75%
Ep 02/10 | Train Loss: 0.1479  Acc: 99.45% | Val Loss:   0.0539  Acc: 99.99%
Ep 03/10 | Train Loss: 0.0357  Acc: 100.00% | Val Loss:   0.0232  Acc: 100.00%
Ep 04/10 | Train Loss: 0.0180  Acc: 100.00% | Val Loss:   0.0139  Acc: 100.00%
Ep 05/10 | Train Loss: 0.0116  Acc: 100.00% | Val Loss:   0.0096  Acc: 100.00%
Ep 06/10 | Train Loss: 0.0083  Acc: 100.00% | Val Loss:   0.0072  Acc: 100.00%
Ep 07/10 | Train Loss: 0.0063  Acc: 100.00% | Val Loss:   0.0057  Acc: 100.00%
Ep 08/10 | Train Loss: 0.0055  Acc: 100.00% | Val Loss:   0.0056  Acc: 100.00%
Ep 09/10 | Train Loss: 0.0054  Acc: 100.00% | Val Loss:   0.0054  Acc: 100.00%
Ep 10/10 | Train Loss: 0.0053  Acc: 100.00% | Val Loss:   0.0053  Acc: 100.00%
Model saved.


# **📌 Train Pair-20 Model**

In [9]:
opt_p20    = Adam(model_p20.parameters(), lr=1e-3)
sched_p20  = StepLR(opt_p20, step_size=7, gamma=0.1)
scaler_p20 = GradScaler()

print("=" * 55)
print("Training: Pair-20 LeNet")
print("=" * 55)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_pair_epoch(model_p20, loader_train_p20,
                                       opt_p20, scaler_p20, criterion, n_q=20)
    vl_loss, vl_acc = eval_pair(model_p20, loader_test_p20, criterion, n_q=20)
    sched_p20.step()
    print(f"Ep {epoch:02d}/{EPOCHS} | "
          f"Train Loss: {tr_loss:.4f}  Acc: {tr_acc*100:.2f}% | "
          f"Val Loss:   {vl_loss:.4f}  Acc: {vl_acc*100:.2f}%")

torch.save(model_p20.state_dict(), "/kaggle/working/lenet_pair20.pth")
print("Model saved.")

Training: Pair-20 LeNet
Ep 01/10 | Train Loss: 1.3510  Acc: 38.97% | Val Loss:   1.1750  Acc: 59.69%
Ep 02/10 | Train Loss: 0.8509  Acc: 69.01% | Val Loss:   0.6351  Acc: 76.58%
Ep 03/10 | Train Loss: 0.5301  Acc: 81.52% | Val Loss:   0.4441  Acc: 85.37%
Ep 04/10 | Train Loss: 0.3521  Acc: 90.32% | Val Loss:   0.2517  Acc: 96.05%
Ep 05/10 | Train Loss: 0.1811  Acc: 98.42% | Val Loss:   0.1361  Acc: 99.21%
Ep 06/10 | Train Loss: 0.1075  Acc: 99.59% | Val Loss:   0.0905  Acc: 99.67%
Ep 07/10 | Train Loss: 0.0737  Acc: 99.85% | Val Loss:   0.0655  Acc: 99.85%
Ep 08/10 | Train Loss: 0.0595  Acc: 99.95% | Val Loss:   0.0627  Acc: 99.86%
Ep 09/10 | Train Loss: 0.0575  Acc: 99.95% | Val Loss:   0.0609  Acc: 99.85%
Ep 10/10 | Train Loss: 0.0558  Acc: 99.96% | Val Loss:   0.0592  Acc: 99.86%
Model saved.


# **📌 Inference Demo**

In [10]:
def predict_single(model, img_path):
    model.eval()
    img = Image.open(img_path).convert("RGB")
    x   = transform_single(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(x)
    pred = logits.argmax(1).item()
    return ['A','B','C','D'][pred]

# Example usage (change path):
# ans = predict_single(model_single, "/kaggle/input/.../Q1/A/0001.png")
# print("Predicted answer:", ans)
print("All 3 models trained and saved to /kaggle/working/")

All 3 models trained and saved to /kaggle/working/
